In [1]:
%load_ext autoreload
%autoreload 2

import scanpy as sc
import sys
import os
import anndata
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import torch
import torch
import torch.distributed as dist
import scipy.sparse as sp
from torch.nn.parallel import DistributedDataParallel as DDP

print("CUDA Available: ", torch.cuda.is_available())
print("Number of GPUs: ", torch.cuda.device_count())


CUDA Available:  True
Number of GPUs:  1


In [ ]:
adata_origin = sc.read_h5ad("../../original_data/plant_all_days.h5ad")

In [3]:

adata_origin.X = adata_origin.X.astype(np.float64)

# X = data.X

# # check the raw data for NaN / Inf before downstream processing
# if sp.issparse(data.X):
#     x_data = data.X.data
#     n_nan = np.isnan(x_data).sum()
#     n_inf = np.isinf(x_data).sum()
#     print("X is sparse")
#     print("NaN in X:", n_nan)
#     print("Inf in X:", n_inf)
#     print("X data min:", np.nanmin(x_data), "max:", np.nanmax(x_data))
# else:
#     n_nan = np.isnan(data.X).sum()
#     n_inf = np.isinf(data.X).sum()
#     print("X is dense")
#     print("NaN in X:", n_nan)
#     print("Inf in X:", n_inf)
#     print("X min:", np.nanmin(data.X), "max:", np.nanmax(data.X))



In [ ]:
for i in [200,500,2000,5000,10000,20000,30000]:
    print("i",i)
    data = adata_origin.copy()
    sc.pp.highly_variable_genes(data, n_top_genes=i)
    data = data[:, data.var.highly_variable].copy()
    group_map = {'Nodule (21 dpi)': 21, 'Nodule (12 dpi)': 12, 'Root': 0}
    data.obs['Group'] = data.obs['Sample_time'].map(group_map)
    # See unique time points
    # print(data.obs['Sample_time'].unique())

    # # See value counts
    # print(data.obs['Sample_time'].unique())

    # print(data.obs['Cell type'].unique())
    print(data)
    # make train data
    plant_data_12 = data[data.obs['Sample_time'] == 'Nodule (12 dpi)'].copy()
    plant_data_21 = data[data.obs['Sample_time'] == 'Nodule (21 dpi)'].copy()
    plant_data_0 = data[data.obs['Sample_time'] == 'Root'].copy() 
    # print(plant_data_12.obs['Sample_time'].unique())
    # print(plant_data_21.obs['Sample_time'].unique())
    # print(plant_data_0.obs['Sample_time'].unique())
    # plant_data_0.write_h5ad(f'plant_paper_day0_adata_gene_top{i}.h5ad')
    # plant_data_12.write_h5ad(f'plant_paper_day12_adata_gene_top{i}.h5ad')
    # plant_data_21.write_h5ad(f'plant_paper_day21_adata_gene_top{i}.h5ad')
    combined = anndata.concat([plant_data_0, plant_data_21])
    print(combined)
    # print(combined.obs['Group'].unique())
    combined.write_h5ad(f'plant_train_for_day12_adata_gene_top{i}.h5ad')
    plant_data_12.write_h5ad(f'plant_real_test_day12_data_gene_top{i}.h5ad')


i 200
AnnData object with n_obs × n_vars = 26712 × 200
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time', 'Group'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock', 'dispersions', 'dispersions_norm'
    uns: 'Cell type_colors', 'hvg'
    obsm: 'X_scvi', 'X_umap'
    layers: 'empty', 'normalize_log', 'raw'
    obsp: 'connectivities', 'distances'
AnnData object with n_obs × n_vars = 18483 × 200
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time', 'Group'
    obsm: 'X_scvi', 'X_umap'
    layers: 'empty', 'normalize_log', 'raw'
i 500
AnnData object with n_obs × n_vars = 26712 × 500
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time', 'Group'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 